# LIMUC structured generative evaluation (Mayo + Evidence)

This notebook evaluates a vision-language model with a structured output format:

- `Mayo: <0|1|2|3>`
- `Evidence: <short visual phrase>`

Use this as a Chapter 4 companion to score-only notebooks:
- `vlm_zero_shot_mayo.ipynb`
- `vlm_lora_finetune_mayo.ipynb`


In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_NO_JAX"] = "1"
os.environ["USE_TF"] = "0"

import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score, mean_absolute_error, mean_squared_error

from transformers import AutoProcessor
try:
    from transformers import AutoModelForVision2Seq
    _HAS_VISION2SEQ = True
except Exception:
    from transformers import Blip2ForConditionalGeneration
    AutoModelForVision2Seq = None
    _HAS_VISION2SEQ = False


In [ ]:
def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
LABEL_MAP_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "label_map.csv"

MODEL_NAME = os.getenv("VLM_MODEL", "Salesforce/blip2-flan-t5-xl")
MAX_SAMPLES = int(os.getenv("MAX_SAMPLES", "0")) or None
SEED = int(os.getenv("SEED", "42"))
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "48"))
PROCESSOR_USE_FAST = int(os.getenv("PROCESSOR_USE_FAST", "1"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "1"))
FORCE_CUDA = int(os.getenv("FORCE_CUDA", "1"))

if FORCE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Disable FORCE_CUDA=1 or use a CUDA-enabled environment.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else None)

PROMPT = (
    "You are a medical endoscopy assistant. Given this colonoscopy image, predict ulcerative colitis severity using Mayo endoscopic score. "
    "Return exactly two lines:\n"
    "Mayo: <0|1|2|3>\n"
    "Evidence: <max 12 words, visual findings only, no treatment advice>."
)

RUBRIC_KEYWORDS = {
    0: ["normal", "vascular", "intact", "no erythema"],
    1: ["mild", "erythema", "decreased vascular"],
    2: ["marked", "erosion", "friability"],
    3: ["ulcer", "bleeding", "spontaneous"],
}
FORBIDDEN_KEYWORDS = ["prednisone", "infliximab", "vedolizumab", "dose", "mg", "prognosis", "surgery"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("DATA_ROOT:", DATA_ROOT)
print("MODEL_NAME:", MODEL_NAME)
print("DEVICE:", DEVICE)
print("PROMPT:", PROMPT)


In [ ]:
meta = pd.read_csv(META_CSV)
images_base = DATA_ROOT / "0_dataset_prep"

def to_abs(p):
    p = Path(p)
    if p.is_absolute():
        return p
    return (images_base / p).resolve()

meta["image_path"] = meta["image_path"].apply(lambda p: str(to_abs(p)))
meta = meta[meta["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

if LABEL_MAP_CSV.exists():
    label_map = pd.read_csv(LABEL_MAP_CSV)
    id_to_name = dict(zip(label_map.label_id, label_map.label_name))
else:
    id_to_name = {i: name for i, name in enumerate(sorted(meta.label_name.unique()))}

name_to_id = {v: k for k, v in id_to_name.items()}
meta["label_id"] = meta["label_name"].map(name_to_id).astype(int)

val_df = meta[meta["split"] == "val"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

if MAX_SAMPLES:
    val_df = val_df.sample(n=min(MAX_SAMPLES, len(val_df)), random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(n=min(MAX_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

processor = AutoProcessor.from_pretrained(MODEL_NAME, use_fast=bool(PROCESSOR_USE_FAST))

if MODEL_DTYPE is not None:
    if _HAS_VISION2SEQ:
        model = AutoModelForVision2Seq.from_pretrained(MODEL_NAME, torch_dtype=MODEL_DTYPE)
    else:
        model = Blip2ForConditionalGeneration.from_pretrained(MODEL_NAME, torch_dtype=MODEL_DTYPE)
else:
    if _HAS_VISION2SEQ:
        model = AutoModelForVision2Seq.from_pretrained(MODEL_NAME)
    else:
        model = Blip2ForConditionalGeneration.from_pretrained(MODEL_NAME)

model = model.to(DEVICE)
model.eval()

print("Val rows:", len(val_df), "| Test rows:", len(test_df))


In [ ]:
def parse_structured_answer(text: str):
    if text is None:
        return None, ""

    m = re.search(r"mayo\s*:\s*([0-3])", text, flags=re.IGNORECASE)
    if not m:
        m = re.search(r"\b([0-3])\b", text)
    mayo = int(m.group(1)) if m else None

    e = re.search(r"evidence\s*:\s*(.*)", text, flags=re.IGNORECASE | re.DOTALL)
    evidence = e.group(1).strip() if e else ""
    if evidence:
        evidence = evidence.split("\n")[0].strip()

    return mayo, evidence

def evidence_checks(mayo, evidence: str):
    evidence_l = (evidence or "").lower()
    word_count = len([w for w in evidence_l.split() if w])
    length_ok = 0 < word_count <= 12
    forbidden = any(k in evidence_l for k in FORBIDDEN_KEYWORDS)
    rubric_hit = False
    if mayo in RUBRIC_KEYWORDS:
        rubric_hit = any(k in evidence_l for k in RUBRIC_KEYWORDS[mayo])
    return {
        "evidence_word_count": word_count,
        "evidence_length_ok": bool(length_ok),
        "evidence_forbidden": bool(forbidden),
        "evidence_rubric_hit": bool(rubric_hit),
    }

def run_split(df: pd.DataFrame):
    rows = []
    for start in range(0, len(df), BATCH_SIZE):
        batch_df = df.iloc[start:start + BATCH_SIZE]
        imgs = [Image.open(p).convert("RGB") for p in batch_df.image_path.values]
        prompts = [PROMPT] * len(imgs)
        inputs = processor(images=imgs, text=prompts, return_tensors="pt", padding=True).to(DEVICE)

        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

        texts = processor.batch_decode(generated, skip_special_tokens=True)

        for row, text in zip(batch_df.itertuples(index=False), texts):
            pred_mayo, evidence = parse_structured_answer(text)
            checks = evidence_checks(pred_mayo, evidence)
            rows.append({
                "split": row.split,
                "img_id": getattr(row, "img_id", None),
                "image_path": row.image_path,
                "y_true": int(row.label_id),
                "y_pred": pred_mayo if pred_mayo is not None else -1,
                "raw_text": text,
                "evidence": evidence,
                **checks,
            })

    return pd.DataFrame(rows)

def summarize(pred_df: pd.DataFrame):
    y_true = pred_df["y_true"].to_numpy(dtype=int)
    y_pred = pred_df["y_pred"].to_numpy(dtype=int)

    valid_mask = np.isin(y_pred, [0, 1, 2, 3])
    parse_rate = float(valid_mask.mean()) if len(y_pred) else 0.0

    summary = {
        "n": int(len(pred_df)),
        "parse_rate": parse_rate,
        "full_accuracy": float((y_true == y_pred).mean()) if len(y_pred) else 0.0,
        "evidence_present_rate": float((pred_df["evidence"].str.len() > 0).mean()) if len(pred_df) else 0.0,
        "evidence_length_ok_rate": float(pred_df["evidence_length_ok"].mean()) if len(pred_df) else 0.0,
        "evidence_forbidden_rate": float(pred_df["evidence_forbidden"].mean()) if len(pred_df) else 0.0,
        "evidence_rubric_hit_rate": float(pred_df["evidence_rubric_hit"].mean()) if len(pred_df) else 0.0,
    }

    if valid_mask.any():
        y_t = y_true[valid_mask]
        y_p = y_pred[valid_mask]
        summary.update({
            "answered_accuracy": float(accuracy_score(y_t, y_p)),
            "answered_balanced_accuracy": float(balanced_accuracy_score(y_t, y_p)),
            "answered_macro_f1": float(f1_score(y_t, y_p, average="macro")),
            "answered_qwk": float(cohen_kappa_score(y_t, y_p, weights="quadratic")),
            "answered_mae": float(mean_absolute_error(y_t, y_p)),
            "answered_rmse": float(np.sqrt(mean_squared_error(y_t, y_p))),
        })
    else:
        summary.update({
            "answered_accuracy": None,
            "answered_balanced_accuracy": None,
            "answered_macro_f1": None,
            "answered_qwk": None,
            "answered_mae": None,
            "answered_rmse": None,
        })

    return summary

val_pred = run_split(val_df)
test_pred = run_split(test_df)

val_summary = summarize(val_pred)
test_summary = summarize(test_pred)

print("VAL SUMMARY")
print(json.dumps(val_summary, indent=2))
print("\nTEST SUMMARY")
print(json.dumps(test_summary, indent=2))

display_cols = ["img_id", "y_true", "y_pred", "evidence", "evidence_word_count", "evidence_length_ok", "evidence_forbidden", "evidence_rubric_hit", "raw_text"]
print("\nVAL SAMPLE")
display(val_pred[display_cols].head(8))
print("\nTEST SAMPLE")
display(test_pred[display_cols].head(8))
